# NGX Long-Term Stock Ranking Model — Research Prototype

**Goal:** rank NGX-listed stocks by likelihood of outperforming the NGX All-Share
Index (relative return) over a 6–12 month forward window, to shortlist long-term
buy candidates for further fundamental review.

**Status of this notebook:** scaffolding + a synthetic data path so every stage
of the pipeline (features → labels → walk-forward training → evaluation) runs
end-to-end today. The real NGX scraper functions are stubbed with the actual
target endpoints documented, ready for you to run locally where you have open
network access (this sandbox can only reach pypi/npm/github, not NGX-facing
sites) — swap `USE_SYNTHETIC_DATA = False` once real data is wired in.

**Data source audit (as of research):**
| Source | What it has | Access |
|---|---|---|
| NGX Group official API | Real-time + historical, index values, corporate actions | Paid, per-product ($1,500–$12,500/yr), contact-to-subscribe |
| NGX historical data (direct) | EOD quotes back to 1996 | Purchase-on-request, 80% student discount |
| afx.kwayisi.org/ngx | Live daily prices, per-stock trading summaries | Free, scrapable HTML, but *not* deep historical — good for building a forward-collected daily series starting now |
| EODHD.com | EOD + fundamentals, JSON/CSV | Paid subscription, has NGX (`.XNSA` suffix) coverage |
| Company annual reports / NGX X-Compliance / Proshare / Nairametrics | Fundamentals, filings | Free but manual/PDF — use the `pdf` extraction skill |
| CBN / Brent crude | Macro overlays (rates, inflation, oil price) | Free, official publications |

**Realistic near-term plan:** since deep historical fundamentals aren't cleanly
available for free, the first real version of this model will likely run on a
**smaller, manually-curated panel** (e.g. top 30–50 liquid NGX names) with
fundamentals hand-extracted from annual reports, rather than all ~150 listed
equities from day one.


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RNG = np.random.default_rng(42)
USE_SYNTHETIC_DATA = True  # flip to False once real collectors are wired in

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


## 1. Data collection

Two collectors: **prices** (daily OHLCV) and **fundamentals** (quarterly/annual,
point-in-time). Both are stubbed here with the real target and parsing notes so
you can drop in `requests`/`BeautifulSoup` logic locally.


In [2]:
def fetch_ngx_daily_prices(tickers, start, end):
    """
    STUB — real implementation target: afx.kwayisi.org/ngx/<ticker>.html
    for live/recent prices, or a paid EODHD.com subscription
    (symbol format e.g. "GTCO.XNSA") for deeper daily history.

    Must run OUTSIDE this sandbox (network here is restricted to
    pypi/npm/github). Suggested approach locally:

        import requests
        from bs4 import BeautifulSoup
        r = requests.get(f"https://afx.kwayisi.org/ngx/{ticker.lower()}.html")
        soup = BeautifulSoup(r.text, "html.parser")
        # parse the trading-summary table -> date, open, high, low, close, volume

    Since that site only exposes a rolling recent window (not deep history),
    the practical pattern is: run this daily/weekly as a cron/scheduled job
    (same pattern you used for the soccer app's Google Sheets logging) and
    accumulate your own historical panel over time, backfilled from EODHD
    or purchased NGX data for the pre-collection period.

    Returns: DataFrame[date, ticker, open, high, low, close, volume]
    """
    raise NotImplementedError("Wire up the real scraper/API call here.")


def fetch_ngx_fundamentals(tickers):
    """
    STUB — real implementation target: annual report PDFs (use the `pdf`
    skill to extract income statement / balance sheet line items) or
    NGX X-Compliance filings, keyed by (ticker, fiscal_period_end,
    publication_date).

    publication_date is critical: fundamentals must be joined into the
    training panel using the date they were PUBLICLY RELEASED, not the
    fiscal period end date, or the model leaks future information (NGX
    filing delays make this an easy trap - some annual reports post
    4-6+ months after year end).

    Returns: DataFrame[ticker, fiscal_period_end, publication_date,
                        eps, revenue, net_income, total_equity, total_debt,
                        dividend_per_share, shares_outstanding, sector]
    """
    raise NotImplementedError("Wire up the real extraction pipeline here.")


In [3]:
def generate_synthetic_ngx_data(n_tickers=40, n_periods=32, seed=42):
    """
    Builds a plausible fake panel (quarterly cadence) so the rest of the
    pipeline is testable today. Structure mirrors what real NGX data will
    look like: ticker x quarter panel with a sector, fundamentals, trailing
    price momentum, and a forward relative return to predict.
    """
    rng = np.random.default_rng(seed)
    sectors = ["Banking", "Oil & Gas", "Consumer Goods", "Insurance", "Industrial"]
    tickers = [f"TCK{i:03d}" for i in range(n_tickers)]
    ticker_sector = {t: rng.choice(sectors) for t in tickers}
    # latent per-ticker "quality" drives both fundamentals and future returns,
    # simulating a real (weak but nonzero) fundamentals -> returns relationship
    quality = {t: rng.normal(0, 1) for t in tickers}

    rows = []
    quarters = pd.period_range("2018Q1", periods=n_periods, freq="Q")
    oil_price_shock = rng.normal(0, 0.05, size=n_periods)  # macro overlay series

    for qi, q in enumerate(quarters):
        for t in tickers:
            sector = ticker_sector[t]
            q_latent = quality[t] + rng.normal(0, 0.5)  # noisy per-quarter signal
            oil_beta = 0.6 if sector == "Oil & Gas" else 0.1
            roe = 0.15 + 0.05 * q_latent + rng.normal(0, 0.03)
            pe = max(2.0, 9 - 2 * q_latent + rng.normal(0, 2))
            div_yield = max(0.0, 0.04 + 0.01 * q_latent + rng.normal(0, 0.01))
            momentum_6m = 0.03 * q_latent + oil_beta * oil_price_shock[qi] + rng.normal(0, 0.08)
            volatility = max(0.05, 0.2 - 0.02 * q_latent + rng.normal(0, 0.03))
            # forward relative return: weak-but-real link to fundamentals/momentum + noise
            fwd_rel_return = (
                0.05 * q_latent + 0.25 * momentum_6m + 0.10 * roe
                - 0.15 * volatility + rng.normal(0, 0.12)
            )
            rows.append(dict(
                ticker=t, quarter=str(q), sector=sector,
                roe=roe, pe=pe, dividend_yield=div_yield,
                momentum_6m=momentum_6m, volatility=volatility,
                oil_price_shock=oil_price_shock[qi],
                fwd_rel_return=fwd_rel_return,
            ))
    return pd.DataFrame(rows)

panel = generate_synthetic_ngx_data() if USE_SYNTHETIC_DATA else None
panel.head()


,ticker,quarter,sector,roe,pe,dividend_yield,momentum_6m,volatility,oil_price_shock,fwd_rel_return
0,TCK000,2018Q1,Banking,0.128,6.418,0.022,-0.126,0.178,-0.084,-0.017
1,TCK001,2018Q1,Insurance,0.082,13.866,0.025,0.017,0.195,-0.084,-0.098
2,TCK002,2018Q1,Insurance,0.177,9.186,0.030,0.049,0.192,-0.084,-0.033
3,TCK003,2018Q1,Consumer Goods,0.108,9.696,0.034,-0.033,0.266,-0.084,-0.110
4,TCK004,2018Q1,Consumer Goods,0.108,11.320,0.044,0.030,0.230,-0.084,0.113


## 2. Feature engineering

Sector-relative z-scoring matters a lot on NGX: banking, oil & gas, and consumer
goods trade on very different fundamental scales, so raw P/E or ROE comparisons
across sectors are misleading. Everything gets z-scored **within sector, within
quarter**.


In [4]:
def add_sector_relative_features(df):
    df = df.copy()
    group = df.groupby(["quarter", "sector"])
    for col in ["roe", "pe", "dividend_yield", "momentum_6m", "volatility"]:
        df[f"{col}_sector_z"] = group[col].transform(
            lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-9)
        )
    return df

panel_feat = add_sector_relative_features(panel)
feature_cols = [c for c in panel_feat.columns if c.endswith("_sector_z")] + ["oil_price_shock"]
panel_feat[feature_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
roe_sector_z,"1,280.000",-0.000,1.000,-2.395,-0.760,-0.012,0.744,2.422
pe_sector_z,"1,280.000",0.000,1.000,-2.750,-0.763,-0.001,0.768,2.378
dividend_yield_sector_z,"1,280.000",0.000,1.000,-2.366,-0.720,-0.062,0.745,2.597
momentum_6m_sector_z,"1,280.000",-0.000,1.000,-2.693,-0.738,0.011,0.729,2.569
volatility_sector_z,"1,280.000",-0.000,1.000,-2.507,-0.758,0.040,0.749,2.424
oil_price_shock,"1,280.000",-0.004,0.033,-0.084,-0.022,-0.002,0.024,0.043


## 3. Labeling

Forward relative return is bucketed into quintiles **within each quarter**
(cross-sectional ranking, not absolute return prediction) — quintile 4 (top 20%)
is the "buy candidate" label.


In [5]:
def add_labels(df):
    df = df.copy()
    df["label_quintile"] = df.groupby("quarter")["fwd_rel_return"].transform(
        lambda s: pd.qcut(s, 5, labels=False, duplicates="drop")
    )
    df["is_top_quintile"] = (df["label_quintile"] == df["label_quintile"].max()).astype(int)
    return df

panel_labeled = add_labels(panel_feat)
panel_labeled[["quarter", "ticker", "fwd_rel_return", "label_quintile"]].tail()


,quarter,ticker,fwd_rel_return,label_quintile
1275,2025Q4,TCK035,-0.010,2
1276,2025Q4,TCK036,0.012,2
1277,2025Q4,TCK037,0.372,4
1278,2025Q4,TCK038,-0.116,1
1279,2025Q4,TCK039,-0.003,2


## 4. Walk-forward split

No random train/test split on panel data — that leaks across time. Each fold
trains on all quarters up to `t`, and validates on quarter `t+1`.


In [6]:
def walk_forward_folds(df, min_train_quarters=8):
    quarters = sorted(df["quarter"].unique())
    for i in range(min_train_quarters, len(quarters) - 1):
        train_q = quarters[:i]
        test_q = quarters[i + 1]
        train = df[df["quarter"].isin(train_q)]
        test = df[df["quarter"] == test_q]
        yield train, test, test_q

fold_count = sum(1 for _ in walk_forward_folds(panel_labeled))
print(f"{fold_count} walk-forward folds")


23 walk-forward folds


## 5. Model training

RandomForest regression on forward relative return as the baseline — same
family of model as the soccer predictor, just applied to a stock panel. Swap in
`LGBMRanker`/`XGBRanker` (LambdaMART) later once real data volume justifies it.


In [7]:
from sklearn.ensemble import RandomForestRegressor

def train_and_predict_fold(train, test, feature_cols):
    model = RandomForestRegressor(
        n_estimators=300, max_depth=4, min_samples_leaf=10,
        random_state=42, n_jobs=-1,
    )
    model.fit(train[feature_cols], train["fwd_rel_return"])
    preds = test.copy()
    preds["pred_score"] = model.predict(test[feature_cols])
    return preds, model

all_fold_preds = []
for train, test, q in walk_forward_folds(panel_labeled):
    preds, _ = train_and_predict_fold(train, test, feature_cols)
    all_fold_preds.append(preds)

oos = pd.concat(all_fold_preds, ignore_index=True)
oos.head()


,ticker,quarter,sector,roe,pe,dividend_yield,momentum_6m,volatility,oil_price_shock,fwd_rel_return,roe_sector_z,pe_sector_z,dividend_yield_sector_z,momentum_6m_sector_z,volatility_sector_z,label_quintile,is_top_quintile,pred_score
0,TCK000,2020Q2,Banking,0.114,6.259,0.028,-0.016,0.238,-0.010,-0.145,-0.569,-1.428,0.021,-0.276,0.610,0,0,-0.004
1,TCK001,2020Q2,Insurance,0.137,7.338,0.057,0.020,0.180,-0.010,-0.020,-0.921,0.096,0.881,0.093,0.359,2,0,-0.029
2,TCK002,2020Q2,Insurance,0.289,3.568,0.059,0.063,0.132,-0.010,0.113,2.101,-1.861,1.040,0.558,-1.484,4,1,0.068
3,TCK003,2020Q2,Consumer Goods,0.177,11.719,0.035,-0.023,0.223,-0.010,-0.099,0.143,1.482,-0.635,-1.002,1.059,1,0,-0.040
4,TCK004,2020Q2,Consumer Goods,0.141,7.329,0.029,0.111,0.198,-0.010,0.202,-0.541,-0.142,-1.176,0.820,0.288,4,1,-0.016


## 6. Evaluation

Not accuracy — this is a ranking task. Metrics:
- **Spearman rank correlation** per quarter (did the model's ranking match actual outperformance ranking?)
- **Precision@top-decile** (of the stocks the model ranked highest, what fraction actually landed in the top quintile of realized returns?)
- **Simple backtest**: equal-weight top-5 picks per quarter vs. the full-panel average return (a stand-in for the NGX-ASI benchmark)


In [8]:
def evaluate(oos):
    per_quarter = []
    for q, g in oos.groupby("quarter"):
        rho, _ = spearmanr(g["pred_score"], g["fwd_rel_return"])
        top_decile_n = max(1, int(len(g) * 0.1))
        top_picks = g.nlargest(top_decile_n, "pred_score")
        precision_top_decile = top_picks["is_top_quintile"].mean()
        top5 = g.nlargest(5, "pred_score")
        backtest_return = top5["fwd_rel_return"].mean()
        benchmark_return = g["fwd_rel_return"].mean()
        per_quarter.append(dict(
            quarter=q, spearman_rho=rho,
            precision_at_top_decile=precision_top_decile,
            top5_return=backtest_return, benchmark_return=benchmark_return,
        ))
    return pd.DataFrame(per_quarter)

results = evaluate(oos)
print("Mean Spearman rho:            ", round(results["spearman_rho"].mean(), 3))
print("Mean precision@top-decile:    ", round(results["precision_at_top_decile"].mean(), 3))
print("Mean top-5 picks return:      ", round(results["top5_return"].mean(), 4))
print("Mean benchmark (all-stock) return:", round(results["benchmark_return"].mean(), 4))
results.tail()


Mean Spearman rho:             0.353
Mean precision@top-decile:     0.424
Mean top-5 picks return:       0.0855
Mean benchmark (all-stock) return: -0.0058


,quarter,spearman_rho,precision_at_top_decile,top5_return,benchmark_return
18,2024Q4,0.346,0.250,0.054,-0.021
19,2025Q1,0.437,0.250,0.057,0.022
20,2025Q2,0.412,0.500,0.112,0.021
21,2025Q3,0.538,0.250,0.077,-0.001
22,2025Q4,0.645,0.750,0.204,-0.017


## 7. Feature importance

Sanity check on what's driving the ranking — with real data this is also a
useful sense-check against domain intuition (e.g. does oil price shock matter
more for Oil & Gas names than for Banking, as economically expected?).


In [9]:
_, last_model = train_and_predict_fold(
    panel_labeled[panel_labeled["quarter"] != sorted(panel_labeled["quarter"].unique())[-1]],
    panel_labeled[panel_labeled["quarter"] == sorted(panel_labeled["quarter"].unique())[-1]],
    feature_cols,
)
importances = pd.Series(last_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances


roe_sector_z              0.462
momentum_6m_sector_z      0.201
pe_sector_z               0.133
dividend_yield_sector_z   0.085
volatility_sector_z       0.081
oil_price_shock           0.038
dtype: float64

## Next steps

1. **Data collection (biggest lift)**: stand up the real `fetch_ngx_daily_prices`
   and `fetch_ngx_fundamentals` collectors locally (outside this sandbox).
   Start with a curated ~30–50 liquid, well-covered tickers rather than the
   full board — matches the realistic data-availability picture above.
2. Backfill 3–5 years of quarterly fundamentals from annual reports (the `pdf`
   skill can help extract line items from report PDFs once you have them).
3. Add macro overlays: Brent crude, USD/NGN, CBN MPR — pulled from free
   official sources, joined on date.
4. Re-run this notebook end-to-end on the real panel (`USE_SYNTHETIC_DATA = False`)
   and sanity-check the evaluation metrics before trusting any rankings.
5. Once real results look stable, this is a natural fit for the Streamlit-app
   treatment you gave the soccer predictor — a "shortlist" dashboard rather
   than a single-number prediction.
